# Go2 + Franka Pick-and-Carry

Genesis の GPU 並列環境で、Go2 の背中に固定した Franka が立方体を目標エリアへ運びます。ランタイムは GPU（T4 / L4）にしてください。ビューアは使いません。

このノートはリポジトリのルート（`pick_and_carry/` と `assets/` が見える場所）で実行します。Colab ならそのディレクトリを `/content/PAI_Task` に置いてください。

In [ ]:
!pip install -q -r requirements.txt
import os
from pathlib import Path
ROOT = Path('/content/PAI_Task') if Path('/content/PAI_Task').exists() else Path.cwd()
os.chdir(ROOT)
print('cwd', Path.cwd())

In [ ]:
from pick_and_carry.config import NUM_ACTIONS, NUM_OBS, OBS_LAYOUT, layout_slices
from pick_and_carry.urdf_tools import validate_urdf
from pick_and_carry.config import default_urdf_path

print('obs', NUM_OBS, 'act', NUM_ACTIONS)
for name, sl in layout_slices(OBS_LAYOUT).items():
    print(f'{sl.start:02d}:{sl.stop:02d}  {name}')
report = validate_urdf(default_urdf_path(), require_actuated_names=True)
print(report['path'])
print('mount', report['mount_parent'], '->', report['mount_child'], report['mount_xyz'])

## 公式 URDF を使うとき

`go2.urdf` と `panda.urdf` をアップロードしてから結合します。未実行でも、下のセルはプリミティブ URDF で動きます。アームが重くて転ぶ場合は `--arm-mass-scale 0.25` を付けて再実行してください。

In [ ]:
import subprocess
from pathlib import Path
go2 = Path('go2.urdf')
panda = Path('panda.urdf')
if go2.is_file() and panda.is_file():
    subprocess.check_call(['python', 'scripts/merge_urdf.py', '--go2', 'go2.urdf', '--panda', 'panda.urdf', '--arm-mass-scale', '0.25'])
else:
    print('go2.urdf / panda.urdf が無いのでプリミティブ URDF を使います')

In [ ]:
import torch
from pick_and_carry.config import EnvConfig
from pick_and_carry.env import PickAndCarryEnv

env = PickAndCarryEnv(cfg=EnvConfig(num_envs=4, backend='gpu', show_viewer=False, seed=0))
obs = env.reset()
action = torch.zeros(env.num_envs, env.num_actions, device=obs.device)
obs, reward, done, info = env.step(action)
print('obs', tuple(obs.shape), 'reward', reward.detach().cpu().tolist())
print('ee link', env.ee_link_name, 'urdf', env.urdf_path)
env.close()

## 短い PPO

初回は Genesis のカーネルコンパイルで止まります。終わると `logs/ppo` に TensorBoard イベントとチェックポイントが残ります。本学習は `--num-envs 256 --iterations 200` のように伸ばします。SAC は `python -m pick_and_carry.train_sac --num-envs 16 --steps 5000` です。

In [ ]:
!python -m pick_and_carry.train_ppo --num-envs 32 --iterations 5 --num-steps 16 --save-interval 5